# Project 1

In [0]:
from pyspark.sql.functions import *

### Step 1:

In [0]:
sales_df=spark.read.csv("/Volumes/dev/demo/circuits/sales.csv", header=True, inferSchema=True)

In [0]:
sales_df.display()

In [0]:
sales_df.printSchema()

In [0]:
sales_df.count()

### Step 2:

In [0]:
print("total number of rows:")
sales_df.count()

In [0]:
print("total number of columns:")
len(sales_df.columns)

In [0]:
print("Total distinct customer")
sales_df.select("customer_id").distinct().count()

In [0]:
print("distinct products")
sales_df.select("product_id").distinct().count()

In [0]:
sales_df.summary().show()

### Step 3:

In [0]:
sales_no_dulpicates_df=sales_df.dropDuplicates()

In [0]:
sales_no_txn_duplicates_df = sales_df.dropDuplicates(["transaction_id"])

In [0]:
sales_no_nulls = sales_df.dropna()

In [0]:
sales_df_clean = (
    sales_df
    .dropDuplicates(["transaction_id"])
    .dropna()
)

### Step 4:

In [0]:
sales_df_clean.printSchema()

In [0]:
sales_df_renamed = sales_df_clean \
    .withColumnRenamed("order_date", "order_dt") \
    .withColumnRenamed("customer_id", "cust_id") \
    .withColumnRenamed("transaction_id", "txn_id") \
    .withColumnRenamed("product_id", "prod_id")

sales_df_renamed.printSchema()

In [0]:
sales_df_derived = sales_df_renamed \
    .withColumn("net_amount", round(col("total_amount") - col("discount_amount"), 2))

In [0]:
sales_df_derived = sales_df_derived \
    .withColumn("order_dt", to_date(col("order_dt"), "yyyy-MM-dd"))

In [0]:
sales_df_final = sales_df_derived \
    .withColumn("load_timestamp", current_timestamp()) \
    .withColumn("load_date", current_date()) \
    .withColumn("file_path", col("_metadata.file_path")) \
    .withColumn("file_name", col("_metadata.file_name"))

In [0]:
sales_df_final.display()

### Step 5:

In [0]:
sales_df_final.write.format("delta").mode("overwrite").saveAsTable("dev.demo.sales_cleaned")

### Step 6:

In [0]:
df =  spark.read.table("dev.demo.sales_cleaned")
df.display()

In [0]:
from pyspark.sql.functions import *

Calculate the total sales.


In [0]:
total_sales_df = df.agg(round(sum("net_amount"), 2).alias("total_sales"))
display(total_sales_df)

# Create a PERMANENT view in dev.demo
spark.sql("""
    CREATE OR REPLACE VIEW dev.demo.total_sales_view AS
    SELECT ROUND(SUM(net_amount), 2) AS total_sales
    FROM dev.demo.sales_cleaned
""")

Find the total number of orders.

In [0]:
total_orders_df=df.agg(countDistinct("order_id").alias(("total_orders")))
display(total_orders_df)

spark.sql("""
    Create Or Replace View dev.demo.total_orders_view As
    Select Count(Distinct order_id) As total_orders
    From dev.demo.sales_cleaned
""")

Find the Total Number of Unique Customers

In [0]:
unique_customer_df=df.agg(countDistinct("cust_id").alias("Unique_customers"))
display(unique_customer_df)

spark.sql("""
    Create Or Replace View dev.demo.unique_customers_view As
    SELECT count(DISTINCT cust_id) AS Unique_customers
    FROM dev.demo.sales_cleaned
""")

Calculate Total Sales for Each Customer

In [0]:
sales_per_customer_df=df.groupBy("cust_id") \
    .agg(round(sum("net_amount"), 2).alias("total_sales")) \
    .orderBy("total_sales", ascending=False)

display(sales_per_customer_df)

spark.sql("""
    CREATE OR REPLACE VIEW dev.demo.sales_per_customer_view AS
    SELECT cust_id, ROUND(SUM(net_amount), 2) AS total_sales
    FROM dev.demo.sales_cleaned
    GROUP BY cust_id
    ORDER BY total_sales DESC
""")

Calculate Total Sales for Each Product

In [0]:
sales_per_product_df = df.groupBy("prod_id") \
    .agg(round(sum("net_amount"), 2).alias("total_sales")) \
    .orderBy("total_sales", ascending=False)
display(sales_per_product_df)

spark.sql("""
    CREATE OR REPLACE VIEW dev.demo.sales_per_product_view AS
    SELECT prod_id, round(sum(net_amount), 2) AS total_sales
    FROM dev.demo.sales_cleaned
    GROUP BY prod_id
    ORDER BY total_sales DESC
""")

Display the Top 10 Customers by Sales

In [0]:
top_3_customers_df=sales_per_customer_df.limit(3)
display(top_3_customers_df)

spark.sql("""
    CREATE OR REPLACE VIEW dev.demo.top_3_customers_view AS
    SELECT *
    FROM dev.demo.sales_per_customer_view
    LIMIT 3
""")

Display the Top 10 Products by Sales

In [0]:
top_3_products_df=df.groupBy("prod_id") \
    .agg(round(sum("net_amount"), 2).alias("total_sales")) \
    .orderBy("total_sales", ascending=False) \
    .limit (3)
display(top_3_products_df)

spark.sql("""
    Create or Replace View dev.demo.top_3_products_view As 
    Select prod_id, round(sum(net_amount), 2) As total_sales
    From dev.demo.sales_cleaned
    Group By prod_id
    order by total_sales desc
    limit 3
""")

Find the Highest Sales Transaction

In [0]:
highest_sale_df = df.orderBy("net_amount", ascending=False).limit(1)
display(highest_sale_df)

spark.sql("""
    CREATE OR REPLACE VIEW dev.demo.highest_sale_view AS
    SELECT * FROM dev.demo.sales_cleaned
    ORDER BY net_amount DESC
    LIMIT 1
""")

Find the Lowest Sales Transaction

In [0]:
lowest_sale_df = df.orderBy("net_amount", ascending=True).limit(1)
display(lowest_sale_df)

spark.sql("""
    CREATE OR REPLACE VIEW dev.demo.lowest_sale_view AS
    SELECT * FROM dev.demo.sales_cleaned
    ORDER BY net_amount ASC
    LIMIT 1
""")

Calculate the Average Order Value

In [0]:
avg_order_value_df = df.agg(round(avg("net_amount"), 2).alias("avg_order_value"))
display(avg_order_value_df)

spark.sql("""
    CREATE OR REPLACE VIEW dev.demo.avg_order_value_view AS
    SELECT ROUND(AVG(net_amount), 2) AS avg_order_value
    FROM dev.demo.sales_cleaned
""")